# 00 — Environment and architecture

**Learning objectives**

- distinguish SedonaDB from SedonaSpark;
- identify every reproducibility dimension before computing a result;
- understand which directories are source, regenerable state, and evidence.

SedonaDB is a single-node Arrow/DataFusion spatial engine. SedonaSpark extends
Spark for distributed spatial processing. The benchmark uses SedonaDB because
it provides a native, multithreaded single-node spatial join comparable with
the existing single-node Kinetica POC. Spark remains a learning environment.

In [ ]:
import json
import os
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import sedona
import sedonadb
import shapely

versions = {
    "platform": platform.platform(),
    "python": platform.python_version(),
    "apache_sedona": sedona.__version__,
    "sedonadb": sedonadb.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
    "shapely": shapely.__version__,
}
pd.Series(versions, name="version").to_frame()

## Data boundaries

`OVERTURE_RELEASE_DIR` is read-only source. `BENCHMARK_DATA_DIR` is
regenerable and may be large. `evidence/` is curated for Git. A credential is
neither data nor configuration: it is mounted separately and is never shown.

In [ ]:
paths = {
    "overture_configured": bool(os.environ.get("OVERTURE_RELEASE_DIR")),
    "benchmark_data_configured": bool(os.environ.get("BENCHMARK_DATA_DIR")),
    "repository": str(Path.cwd()),
}
print(json.dumps(paths, indent=2))

## Architecture checkpoint

The fixed data flows to both engines. Engine output returns to one
correctness gate before performance is interpreted. This prevents a fast but
semantically different query from becoming a headline number.

**Exercise:** identify which manifest fields should change if the same data is
measured on a second host. CPU topology and image digest may change; source,
configuration, and canonical checksums should not.